In [ ]:
import numpy as np
import cv2
import time
import matplotlib.pyplot as plt
from PIL import Image
import os

In [ ]:
angle_range = [-45, 45] # 角度范围，单位度
angle_resolution = 1 # 角度分辨率
aug_num = 60 # 每张图片增强多少张
use_bg = False # 是否使用背景

src_img_path = r"C:\Users\hp\Desktop\fall\all1\leftImg8bit"
src_mask_path = r"C:\Users\hp\Desktop\fall\all1\annotations"
src_bg_path = r"C:\Users\hp\Desktop\fall\all1\bg" # 背景图片名字必须与原图片一致

save_img_path = r"C:\Users\hp\Desktop\fall_aug\leftImg8bit"
save_mask_path = r"C:\Users\hp\Desktop\fall_aug\gtFine"

In [ ]:
angles_list = np.linspace(angle_range[0], angle_range[1], int((angle_range[1]-angle_range[0])/float(angle_resolution))) # 旋转角度的范围
angles_list_no = np.linspace(0,0,4) # 四张原图

In [ ]:
def get_color_map_list(num_classes):
    """
    Returns the color map for visualizing the segmentation mask,
    which can support arbitrary number of classes.
    Args:
        num_classes (int): Number of classes.
    Returns:
        (list). The color map.
    """

    num_classes += 1
    color_map = num_classes * [0, 0, 0]
    for i in range(0, num_classes):
        j = 0
        lab = i
        while lab:
            color_map[i * 3] |= (((lab >> 0) & 1) << (7 - j))
            color_map[i * 3 + 1] |= (((lab >> 1) & 1) << (7 - j))
            color_map[i * 3 + 2] |= (((lab >> 2) & 1) << (7 - j))
            j += 1
            lab >>= 3
    color_map = color_map[3:]
    return color_map

In [ ]:
color_map = get_color_map_list(256)

In [ ]:
#创建文件夹
if not os.path.exists(save_img_path):
    os.makedirs(save_img_path)
if not os.path.exists(save_mask_path):
    os.makedirs(save_mask_path)

In [ ]:
imgs_name = []
for _item in os.listdir(src_img_path):
    if _item.split('.')[-1] in ['jpg', 'png', 'bmp', 'jpeg']:
        imgs_name.append(_item)

In [ ]:
def rotate_and_resize_roi(img1, theta, scale_ratio=1):
    img = img1.copy()
    _rotateCenter = (np.shape(img)[1]//2, np.shape(img)[0]//2)#旋转中心
    _img_size = (np.shape(img)[1], np.shape(img)[0])#变换后的大小
    w,h = _img_size[0], _img_size[1]
    alpha = abs(theta)
    ratio_app = (w/2.0 / np.cos((45-alpha)*np.pi/180))/np.sqrt((w/2.0)**2+(h/2.0)**2)
    scale_ratio = scale_ratio*ratio_app
    _R = cv2.getRotationMatrix2D(_rotateCenter, theta, scale_ratio) #计算旋转的仿射变换矩阵
    img_rotate = cv2.warpAffine(img, _R, _img_size)
    return theta, scale_ratio, img_rotate

In [ ]:
# 原图片本身旋转
angles_list = np.hstack([angles_list, angles_list_no])
angles_list = list(angles_list.astype(np.int32))

for _imgName in imgs_name:
    _maskName = _imgName.split('.')[0] + '.png'
    print("start deal "+ _imgName)
    angles_list_copy = angles_list.copy()
    for i in range(0, aug_num):
        angle = np.random.choice(angles_list_copy)
        if(len(angles_list)>1):
            angles_list_copy.remove(angle) # 不重复角度
        scale = 1
        img = Image.open(os.path.join(src_img_path, _imgName))#加载图片
        img_mask = Image.open(os.path.join(src_mask_path, _maskName))
        if(use_bg==True):
            img_bg = Image.open(os.path.join(src_bg_path, _imgName))#加载图片
        
        img = np.array(img)
        img_mask = np.array(img_mask)
        if(use_bg==True):
            img_bg = np.array(img_bg)

        _theta, _scale, img2_arr = rotate_and_resize_roi(img, angle, scale)
        _theta, _scale, img_mask2 = rotate_and_resize_roi(img_mask, angle, scale)

        if(use_bg==True):
            # R,G,B的值均为0才是黑色
            img2_arr_mask = np.logical_and(img2_arr[:,:,0] == 0 , img2_arr[:,:,1] == 0) # 求R和G通道bool矩阵的交集
            img2_arr_mask = np.logical_and(img2_arr_mask, img2_arr[:,:,2] == 0)
            # img2_arr[img2_arr_mask]= np.array([255,255,122], dtype=np.uint8)
            img2_arr[img2_arr_mask]= img_bg[img2_arr_mask]
        
        # 保存新图片和标签
        _new_img_name = _imgName.split('.')[0]+'_'+str(i)+'_'+str(angle)+'deg'+'.'+_imgName.split('.')[-1]
        converted_img = Image.fromarray(img2_arr)
        converted_img.save(os.path.join(save_img_path, _new_img_name))

        lbl_pil = Image.fromarray(img_mask2)
        lbl_pil.putpalette(color_map)
        _new_mask_name = _maskName.split('.')[0]+'_'+str(i)+'_'+str(angle)+'deg'+'.'+_maskName.split('.')[-1]
        lbl_pil.save(os.path.join(save_mask_path, _new_mask_name))
    
print('end--------------------------')